# DataAnalysisAgent - 智能数据分析助手

## 第1部分：环境配置

In [2]:
# 直接安装依赖库，无需读取外部文件
# 强制指定 numpy 版本小于 2.0
!pip install "numpy<2.0.0" "hello-agents[all]>=0.1.0" "xlrd>=2.0.1" "python-dotenv>=1.0.0" "jupyter>=1.0.0" "notebook>=7.0.0" tabulate
import os
import io
import contextlib
import traceback
import pandas as pd
import json
import re
from typing import Dict, Any, List
# 尝试导入 ReActAgent，如果不可用则回退到 SimpleAgent
try:
    from hello_agents import ReActAgent as AgentClass
    print("✅ 使用 ReActAgent")
except ImportError:
    from hello_agents import SimpleAgent as AgentClass
    print("⚠️ ReActAgent 不可用，回退到 SimpleAgent")

from hello_agents import HelloAgentsLLM
from hello_agents.tools import Tool, ToolParameter

# --- 步骤 1：更新 API 配置（切换为 DeepSeek）---
os.environ["LLM_MODEL_ID"] = "deepseek-v4-flash"
# 请通过环境变量或 .env 文件提供 API Key，不要在 notebook 中硬编码密钥
os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com/v1"
os.environ["LLM_TIMEOUT"] = "120" # 增加超时时间以适应复杂推理
# ================= 关键修复 =================
import hello_agents.context.token_counter
import tiktoken

# 强制将 TokenCounter 内部获取编码器的方法 Patch 掉
# 无论什么模型，都强制使用 cl100k_base (GPT-4 标准编码器)
# 这样可以绕过 deepseek-chat 不被识别的问题
def patched_get_encoding(self):
    return tiktoken.get_encoding("cl100k_base")

hello_agents.context.token_counter.TokenCounter._get_encoding = patched_get_encoding
print("✅ 已应用 TokenCounter 补丁 (DeepSeek 适配)")
# ===========================================
print("✅ 库导入和 DeepSeek 配置完成")

✅ 使用 ReActAgent


ModuleNotFoundError: No module named 'hello_agents.context'

## 第2部分：定义数据分析工具

In [6]:
class PythonInterpreterTool(Tool):
    """Python 代码解释器 - 作为一个沙箱执行动态生成的 Python 代码"""
    
    def __init__(self):
        super().__init__(
            name="PythonInterpreterTool",
            description="执行 Python 代码并返回打印输出（stdout）。"
        )
        # 为沙箱准备安全的全局变量
        self.globals = {
            "pd": pd,
            "json": json,
            "sys": sys,
            "os": os
        }

    def run(self, parameters: Dict[str, Any]) -> str:
        """
        执行传入的 Python 代码。
        parameters:
            code: str, 需要执行的 Python 代码
        """
        code = parameters.get("code", "")
        if not code:
            return "错误：未提供代码"
        
        # 捕获标准输出
        old_stdout = sys.stdout
        redirected_output = io.StringIO()
        sys.stdout = redirected_output
        
        try:
            # 在沙箱中执行代码
            exec(code, self.globals)
            output = redirected_output.getvalue()
            return output if output.strip() else "代码执行成功，无输出。"
        except Exception:
            # 捕获并返回完整的错误堆栈
            error_msg = traceback.format_exc()
            return f"代码执行出错:\n{error_msg}"
        finally:
            sys.stdout = old_stdout

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="code",
                type="string",
                description="要执行的 Python 代码片段",
                required=True
            )
        ]

print("✅ PythonInterpreterTool 定义完成")

✅ PythonInterpreterTool 定义完成


## 第3部分：创建智能体

In [7]:
from hello_agents import ToolRegistry

# 定义工具集
tool_registry = ToolRegistry()
tool_registry.register_tool(PythonInterpreterTool(), "python_interpreter_tool")

# --- 步骤 4：Agent 大脑升级与 Prompt 重构 ---
system_prompt = """你是一个严谨的数据科学家。你现在的任务是基于提供的本地数据集进行深度分析。
你拥有一个强大的 Python 解释器工具 `PythonInterpreterTool`，可以执行任意 Python 代码。

你的工作流程如下：
1. **理解数据**：首先查看用户提供的数据集概要信息（列名、样本），明确分析目标。
2. **编写代码**：不要凭空猜测数据特征。必须编写 Python 代码（使用 pandas, numpy, scipy, matplotlib, seaborn 等库）来读取数据、清洗数据、统计描述、绘制图表或进行假设检验。
   - 数据文件路径固定为：`./data/simple_data.xls`
   - 如果需要绘图，请使用 `matplotlib` 或 `searborn` 并将图表保存到 `./output/` 目录，或者生成 ECharts 代码。
3. **执行与修正**：调用 `PythonInterpreterTool` 执行代码。
   - 如果代码报错，请阅读错误信息并编写新的代码进行修复。
   - 如果执行成功但结果不完整，继续编写后续代码。
4. **得出结论**：基于代码执行的客观结果（Observation），撰写最终的分析报告。

**重要约束**：
- 在分析过程中，凡是涉及数据计算、统计、绘图，**必须** 使用 Python 代码完成，禁止直接杜撰数字。
- 最终报告必须包含专业的统计学结论（如 P 值、相关性分析等），并以 Markdown 格式输出。
- 如果需要清洗数据（如处理缺失值），请在代码中完成并打印处理后的数据摘要。

请保持思考的逻辑性（Thought -> Action -> Observation -> Thought...）。
当分析完成时，请输出最终的 Markdown 报告。
"""

# 创建智能体
agent = AgentClass(
    name="高级数据分析师",
    llm=HelloAgentsLLM(),
    system_prompt=system_prompt,
    tool_registry=tool_registry
)

print(f"✅ 智能体创建完成 (类型: {agent.__class__.__name__})")
print(f"✅ 可用工具: {list(tool_registry._tools.keys())}")

NameError: name 'sys' is not defined

## 第4部分：读取示例数据表格

In [8]:
file_path = "C:\\Users\\pc\\OneDrive\\Desktop\\agent\\data\\simple_data.xls"

data_context = ""

try:
    df = pd.read_excel(file_path)
    
    # 提取关键信息，解决 Prompt 过长的问题
    col_names = list(df.columns)
    sample_data = df.head(5).to_markdown()
    shape = df.shape
    
    data_context = f"""
    数据集文件路径: {file_path}
    列名: {col_names}
    数据形状: {shape}
    前5行样本数据:
    {sample_data}
    """
    
    print("✅ 数据信息提取成功:")
    print(data_context)

except FileNotFoundError:
    data_context = f"错误：文件 {file_path} 未找到"
except Exception as e:
    data_context = f"错误：读取文件失败 {str(e)}"
    traceback.print_exc()

✅ 数据信息提取成功:

    数据集文件路径: C:\Users\pc\OneDrive\Desktop\agent\data\simple_data.xls
    列名: ['指标', '2025年10月', '2025年9月', '2025年8月', '2025年7月', '2025年6月']
    数据形状: (13, 6)
    前5行样本数据:
    |    | 指标                                           |   2025年10月 |   2025年9月 |   2025年8月 |   2025年7月 |   2025年6月 |
|---:|:-----------------------------------------------|-------------:|------------:|------------:|------------:|------------:|
|  0 | 居民消费价格指数(上年同月=100)                 |        100.2 |        99.7 |        99.6 |       100   |       100.1 |
|  1 | 食品烟酒类居民消费价格指数(上年同月=100)       |         98.4 |        97.4 |        97.5 |        99.2 |       100.1 |
|  2 | 衣着类居民消费价格指数(上年同月=100)           |        101.7 |       101.7 |       101.8 |       101.7 |       101.6 |
|  3 | 居住类居民消费价格指数(上年同月=100)           |        100.1 |       100.1 |       100.1 |       100.1 |       100.1 |
|  4 | 生活用品及服务类居民消费价格指数(上年同月=100) |        101.9 |       102.2 |       101.8 |       101.2 |       100.7 |
    


## 第5部分：执行数据分析

In [9]:
# 执行高级数据分析
print("=== 开始数据分析 ===")
instruction = f"对以下数据集进行深入分析，并撰写数据分析报告。\n\n{data_context}\n"
result = agent.run(instruction)

print("✅ 分析执行完成")
print(result)

=== 开始数据分析 ===


NameError: name 'agent' is not defined

## 第6部分：保存分析报告和图表

In [14]:
import re
import os

echarts_match = re.search(r"option\s*=\s*(\{[\s\S]*?\});", result)
if echarts_match:
    echarts_code = echarts_match.group(1)
    print(echarts_code)
else:
    print("未找到 ECharts 代码")

{
    xAxis: {
        type: 'category',
        data: ['2025年6月', '2025年7月', '2025年8月', '2025年9月', '2025年10月']
    },
    yAxis: {
        type: 'value'
    },
    series: [
        {
            name: '居民消费价格指数',
            data: [100.1, 100.0, 99.6, 99.7, 100.2],
            type: 'line'
        },
        {
            name: '食品烟酒类居民消费价格指数',
            data: [100.1, 99.2, 97.5, 97.4, 98.4],
            type: 'line'
        },
        {
            name: '衣着类居民消费价格指数',
            data: [101.6, 101.7, 101.8, 101.7, 101.7],
            type: 'line'
        },
        {
            name: '居住类居民消费价格指数',
            data: [100.1, 100.1, 100.1, 100.1, 100.1],
            type: 'line'
        },
        {
            name: '生活用品及服务类居民消费价格指数',
            data: [100.7, 101.2, 101.8, 102.2, 101.9],
            type: 'line'
        },
        {
            name: '交通通信类居民消费价格指数',
            data: [96.3, 96.9, 97.6, 98.0, 98.5],
            type: 'line'
        },
        {
            name

In [8]:
report_match = re.search(r"(# 数据分析报告[\s\S]*)", result)

markdown_report = report_match.group(1).strip()
print("提取到 Markdown 报告")


# ==============================
# 3. 保存 Markdown 报告到文件
# ==============================
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)
md_path = os.path.join(output_dir, "report.md")

with open(md_path, "w", encoding="utf-8") as f:
    f.write(markdown_report)

print(f"\nMarkdown 报告已保存至: {md_path}")

提取到 Markdown 报告

Markdown 报告已保存至: ./output\report.md


In [13]:
from IPython.display import HTML

html_code = f'''
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>第一个 ECharts 实例</title>
    <!-- 引入 echarts.js -->
    <script src="https://cdn.staticfile.org/echarts/4.3.0/echarts.min.js"></script>
</head>
<body>
    <!-- 为ECharts准备一个具备大小（宽高）的Dom -->
    <div id="main" style="width: 600px;height:400px;"></div>
    <script type="text/javascript">
        // 基于准备好的dom，初始化echarts实例
        var myChart = echarts.init(document.getElementById('main'));
 
        // 指定图表的配置项和数据
        var option = {echarts_code}
        
        // 使用刚指定的配置项和数据显示图表。
        myChart.setOption(option);
    </script>
</body>
</html>
'''

In [15]:
from IPython.display import IFrame

# 将 HTML 代码保存为文件
with open('./output/echarts.html', 'w', encoding='utf-8') as f:
    f.write(html_code)
